# 📘 Text Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/openai-api-hands-on/blob/main/1_Text_Generation/01-Text%20Generation-Lecture.ipynb)

> **주의:** Colab 런타임을 재시작하면 환경 변수가 초기화되므로 `OPENAI_API_KEY`를 다시 설정해야 합니다.

## 1) 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U openai

### API 키 설정 및 Client 생성
- **Colab**: 왼쪽 메뉴 `보안 비밀(열쇠 아이콘)`에 `OPENAI_API_KEY`를 저장하고 **노트북 액세스**를 켜 두면 자동으로 불러옵니다.
- **로컬**: 환경 변수 `OPENAI_API_KEY`를 미리 설정해 두거나, 아래 셀을 실행했을 때 뜨는 입력창에 키를 붙여넣으면 됩니다.
- 아래의 코드에서 오류가 난다면 API 키의 오류일 가능성이 높습니다.

In [1]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

✅ API 키 설정 완료


In [2]:
base_dirs = [
    ("data", ["image", "audio", "video"]),
    ("results", ["image", "audio", "video"]),
]

for root, subfolders in base_dirs:
    for sub in subfolders:
        path = os.path.join(root, sub)
        os.makedirs(path, exist_ok=True)
        print(f"ready: {path}")

ready: data/image
ready: data/audio
ready: data/video
ready: results/image
ready: results/audio
ready: results/video


In [3]:
from openai import OpenAI

client = OpenAI()
print("✅ 클라이언트 생성 완료")

✅ 클라이언트 생성 완료


### OpenAI API 시작하기
- Responses API를 사용하면 챗GPT처럼 프롬프트에 따라 다양한 텍스트(코드, 수식, JSON, 자연어 등)를 간편하게 만들 수 있습니다.

In [4]:
response = client.responses.create(
    model="gpt-6-luna",
    instructions="You are a helpful assistant. Reply in Korean.",
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

달빛을 따라 별구름 위를 산책한 유니콘은 아기 별들에게 자장가를 불러 주고 포근한 꿈속으로 잠들었어요.


- 모델 출력(response) 예시
    - `output_text` 말고도 모델 이름, 사용한 파라미터(`reasoning`, `text`), 토큰 사용량(`usage`) 등 많은 정보가 함께 들어 있습니다.

In [5]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_01bb933877895c44006adf666b713087d28aa8fa2ffc2fef1b",
  "access_programs": null,
  "created_at": 1793025643.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "You are a helpful assistant. Reply in Korean.",
  "metadata": {},
  "model": "gpt-6-luna",
  "object": "response",
  "output": [
    {
      "id": "rs_01bb933877895c44006adf666c98a087d2b31df0b9579ac42c",
      "summary": [],
      "type": "reasoning",
      "content": [],
      "encrypted_content": "gAAAAABq32Ztmu1cp5o4VzZMGw0uabDMeXqxglBbKLue_aoAeZkifzVNxlE01xNjP0UkodpGCbGTH5Hr5aRrTltQIaq_7GqaPjcRionmIQ7zaMScLB_G--9b5Kgm1EV9Odw9GmeUZ9dSOnYk9jxfkhXtTFP_N3dsJOr8Qf1PZegTT7rxBiJMZ_D1hLVMxq5ovpjnh0DeiwR652q5Sj26mbJjD5wl4jrUpRSBMkP6g-c2JviCX0QTOgt6K8uGv4XA3F_CugF-NWM6r91hUXAJz0-vD81u2BNP9JNVjnfsvd6mgeZtFYGOKcT7h4N4oEKHeOXFZ3r9oJT4sULZa4f9uEVjP24GDKoEGJoNC6S1BASa6VzMHzHF7Cr-foohvPdHBaZoPi_GKEeDnzacVBoCPT-m8Zl9Z81X4AXYRJFavAa3nbpvNY3wFoaNN6-pJQ55jqTIcEocq0405D3bGle9S2YXbNqR7aEnE0vQqzeJSJROJ8jyo1RiJE6aieG

모델이 생성한 콘텐츠는 `response.output` 속성에서 확인할 수 있습니다.
- 도구 호출, reasoning 토큰 데이터 등 여러 항목 포함됨
- 그래서 `output[0]`이 항상 텍스트라고 가정하면 안 되고, 텍스트만 필요할 땐 `response.output_text`를 쓰는 게 편합니다.

In [6]:
response.output

[ResponseReasoningItem(id='rs_01bb933877895c44006adf666c98a087d2b31df0b9579ac42c', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq32Ztmu1cp5o4VzZMGw0uabDMeXqxglBbKLue_aoAeZkifzVNxlE01xNjP0UkodpGCbGTH5Hr5aRrTltQIaq_7GqaPjcRionmIQ7zaMScLB_G--9b5Kgm1EV9Odw9GmeUZ9dSOnYk9jxfkhXtTFP_N3dsJOr8Qf1PZegTT7rxBiJMZ_D1hLVMxq5ovpjnh0DeiwR652q5Sj26mbJjD5wl4jrUpRSBMkP6g-c2JviCX0QTOgt6K8uGv4XA3F_CugF-NWM6r91hUXAJz0-vD81u2BNP9JNVjnfsvd6mgeZtFYGOKcT7h4N4oEKHeOXFZ3r9oJT4sULZa4f9uEVjP24GDKoEGJoNC6S1BASa6VzMHzHF7Cr-foohvPdHBaZoPi_GKEeDnzacVBoCPT-m8Zl9Z81X4AXYRJFavAa3nbpvNY3wFoaNN6-pJQ55jqTIcEocq0405D3bGle9S2YXbNqR7aEnE0vQqzeJSJROJ8jyo1RiJE6aieGPOkIz6L8Rs-WEhRW3j3tqoEBQQIme7VqmlD4y_5kUB5biI0oOXcm9KinBCOkfZOfJ0kcZfv8K2kxx9cjvM2bU3NcRRLGJab6UwpFP2t5qAYIjVjsCOEdnH7puuMrHwkCHDJ-PSJWtS21WkNj5sHptvQGSS-eMdTz0_t0FZ-6LznqhQL8mdXOj3owwtioKfKqQ0EGVmpD_j4R5ibDJ8117nFIfrVLqupjmDmR_2oSMx75g7SJonovRntngvu0cy3Eaa4J3HbHDZVKhFotNNRZstKRGtyh5qQcwTO4ua60lOTtb1tJB7eMSyfeefZISyU3TRIzYlGmJdiTXW5tvIu-Et4J4CBlt

## 2) 주요 파라미터

### model
- 응답을 생성할 모델 ID(`gpt-6-astra`, `gpt-6.1-sol`, `gpt-6-luna` 등)를 지정합니다.
- 모델별 성능과 비용은 [OpenAI 모델 가이드](https://platform.openai.com/docs/models)에서 확인할 수 있습니다.

### instructions
- 모델 컨텍스트에 넣는 시스템/개발자 지침으로, 응답의 톤·역할·제약을 정의합니다.
- `previous_response_id`와 함께 사용하면 이전 턴 instructions를 끊고 새 instruction으로 쉽게 교체할 수 있습니다.
    - `previous_response_id`: 이전 응답의 고유 ID입니다. 이 값을 활용하면 멀티턴 대화를 구성할 수 있습니다.
- `instructions`는 `input` 파라미터보다 높은 우선순위를 가집니다.

### input
- 텍스트, 이미지, 파일 등 모델이 답변을 생성할 때 참고하는 입력값입니다.

### max_output_tokens
- 가시 출력 토큰(visible output token)과 추론 토큰(reasoning token)을 모두 포함한 최대 생성 길이를 설정합니다.
- 값이 너무 작으면 답변이 끝나기 전에 잘려서 `status`가 `incomplete`가 되고, 추론하다가 토큰을 다 쓰면 **답변이 아예 비어 있을 수도** 있습니다.

In [7]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
        model="gpt-6-luna",
        instructions="You are a helpful assistant. Reply in Korean.",
        input="Write a one-sentence bedtime story about a unicorn.",
    )

print(response.output_text)

달빛을 머금은 유니콘은 별빛 숲에서 아기 토끼들을 재워 준 뒤, 포근한 구름 위에서 행복한 꿈을 꾸었어요.


In [8]:
from openai import OpenAI

client = OpenAI()

# max_output_tokens 예제
for tokens in [16, 300, 1000]:
    response = client.responses.create(
        model="gpt-6-luna",
        instructions="You are a helpful assistant. Reply in Korean.",
        input="Write a one-sentence bedtime story about a unicorn.",
        max_output_tokens=tokens
    )
    print(f"\n--- max_output_tokens={tokens} (status: {response.status}) ---")
    if response.status == "incomplete":
        print("⚠️ 중단 사유:", response.incomplete_details.reason)
    print(response.output_text)


--- max_output_tokens=16 (status: incomplete) ---
⚠️ 중단 사유: max_output_tokens




--- max_output_tokens=300 (status: completed) ---
달빛 아래 잠든 숲을 지나던 유니콘은 별 하나를 뿔에 살며시 얹고, 모두에게 달콤한 꿈을 선물했어요.



--- max_output_tokens=1000 (status: completed) ---
달빛이 은은히 내리는 숲에서 유니콘은 별들에게 자장가를 불러 주고 포근한 구름 이불 속에서 잠이 들었어요.


## 3) 모델별 출력 차이 비교 실습

GPT-6 시리즈는 총 세 가지 모델이 있습니다. (2026년 10월 기준)

| 모델 | 주요 용도 | 가격 (1M 토큰, 입력 / 출력) |
|-----------|-----------------------------------------------------|------------------|
| gpt-6-astra | 가장 똑똑한 모델. 복잡한 추론, 코드 작성, 다단계 작업 등 어려운 task에 적합 | $10 / $50 |
| gpt-6.1-sol | 지능·속도·비용의 균형. 채팅·일반적 task에 적합 | $2 / $10 |
| gpt-6-luna | 가장 빠르고 저렴. 단순 분류·지시 수행 등 대량 처리에 적합 | $0.10 / $0.50 |

> 💡 2025년 수업 당시에는 `gpt-5` / `gpt-5-mini` / `gpt-5-nano` 조합이었는데, 이 모델들은 2026년 12월 11일에 종료됩니다.
> 모델 이름은 계속 바뀌니까 실습할 때는 아래 링크에서 최신 모델을 한 번 확인해 보세요.

OpenAI 제공 모델 리스트
- 링크: https://platform.openai.com/docs/models

In [9]:
import time
from openai import OpenAI

client = OpenAI()

llm_prompt = "대형 언어 모델(LLM)이 무엇인지 3문장으로 설명하세요."

for model in ["gpt-6-astra", "gpt-6.1-sol", "gpt-6-luna"]:
    start = time.time()
    response = client.responses.create(
        model=model,
        instructions="You are a helpful instructor. Reply in Korean and keep the answer well-structured.",
        input=llm_prompt,
    )
    elapsed = time.time() - start
    print(f"===== {model} ({elapsed:.1f}초, 출력 토큰 {response.usage.output_tokens}개) =====")
    print(response.output_text, end="\n\n")

===== gpt-6-astra (5.3초, 출력 토큰 118개) =====
대형 언어 모델(LLM)은 방대한 텍스트 데이터를 학습해 인간의 언어를 이해하고 생성하도록 설계된 인공지능 모델입니다.
문맥을 바탕으로 다음에 올 단어나 토큰을 예측하는 방식으로 작동하며, 질문 답변·요약·번역·글쓰기 등 다양한 작업을 수행합니다.
다만 사실과 다른 내용을 그럴듯하게 만들어 내거나 학습 데이터의 편향을 반영할 수 있으므로, 중요한 정보는 별도로 확인해야 합니다.



===== gpt-6.1-sol (5.8초, 출력 토큰 116개) =====
대형 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여 사람의 언어를 이해하고 생성하도록 설계된 인공지능 모델입니다.
문맥을 바탕으로 다음에 올 단어 또는 단어 조각을 예측하는 방식으로 질문 답변, 글쓰기, 번역, 요약 등 다양한 작업을 수행합니다.
다만 사실과 다른 내용을 그럴듯하게 생성하거나 학습 데이터의 편향을 반영할 수 있으므로, 중요한 정보는 별도로 확인해야 합니다.



===== gpt-6-luna (1.9초, 출력 토큰 125개) =====
대형 언어 모델(LLM)은 방대한 텍스트 데이터를 학습해 언어의 패턴과 맥락을 익힌 인공지능 모델입니다. 질문에 답하거나 글을 쓰고, 번역·요약·코드 작성 같은 언어 관련 작업을 수행할 수 있습니다. 다만 학습 데이터의 한계나 잘못된 추론으로 부정확한 답을 할 수 있으므로 결과를 확인하는 것이 좋습니다.



## 4) reasoning.effort · text.verbosity · 메시지 역할

### reasoning.effort
- 모델이 응답을 생성하기 전 얼마나 많은 추론 토큰을 사용할지 결정합니다.
- reasoning effort는 `none`, `low`, `medium`, `high`, `xhigh`, `max`를 지원합니다. (모델마다 지원 범위가 조금씩 다름)
    - `none`: 추론 없이 바로 답변 (`gpt-6-luna` 등 일부 모델만 지원)
    - GPT-5 때 있던 `minimal`은 GPT-6 계열에서 지원하지 않습니다. → 대신 `low` 사용
- default: medium
- 💡 추론(reasoning)을 사용하는 동안에는 `temperature`, `top_p` 같은 샘플링 파라미터를 쓸 수 없습니다.

In [10]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6-luna",
    input="남산타워 외벽을 1mm 두께의 금으로 도금하려면 금이 몇 kg이나 필요할까요?",
    reasoning={
        "effort": "high" # none, low, medium, high, xhigh, max
    }
)

print(response.output_text)
print("\n🧠 사용한 추론 토큰 수:", response.usage.output_tokens_details.reasoning_tokens)

대략 **10만~14만 kg, 즉 100~140톤** 정도로 추정할 수 있습니다.

금의 밀도는 약 **19,320 kg/m³**이므로, 두께 1mm로 바르면 면적 1m²당 금이 약 **19.3kg** 필요합니다. 남산서울타워의 안테나를 제외한 건물 외벽 면적을 대략 **5,000~7,000m²**로 잡으면:

**5,000~7,000m² × 19.3kg/m² ≈ 97,000~135,000kg**

외벽 면적과 실제 형태에 따라 달라지는 거친 추정치입니다. 안테나까지 높이 전체를 속이 찬 원통처럼 가정하면 계산값은 더 커지지만, 실제 안테나는 외벽이 있는 건물이 아니라서 그렇게 계산하는 건 적절하지 않습니다.

🧠 사용한 추론 토큰 수: 1880


### text.verbosity
- 출력 답변의 길이 (토큰 수)에 영향을 주는 파라미터
- verbosity: 다변, 수다, 장황
- verbosity는 low, medium, high 옵션을 지원합니다.
   - low: 짧고, 간결하며 요점만 전달 (예: 요약, 코드 스니펫)
   - medium: 적절한 상세 설명과 길이 (기본값)
   - high: 길고, 자세하며 세부 설명이 많은 답변 제공

In [11]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6-luna",
    input="삶, 우주 그리고 모든 것에 대한 궁극적인 질문의 답은 무엇인가요?",
    text={
        "verbosity": "low" # medium, high
    }
)

print(response.output_text)

42입니다. 다만 궁극적인 질문이 무엇인지는 아직 밝혀지지 않았죠.


### 메시지 역할(role)
| 역할(role)     | 설명                                                    | 우선순위                                 |
| -------------- | ------------------------------------------------------- | ---------------------------------------- |
| developer      | 애플리케이션 개발자가 제공하는 instructions             | 가장 높음 (user 메시지보다 우선)         |
| user           | 사용자가 제공하는 instructions                        | developer 메시지보다 낮고 assistant보다 높음 |
| assistant      | 모델이 생성하는 응답 메시지                              | 가장 낮음                                |

- OpenAI 모델 스펙에서는 역할 우선순위를 `developer` > `user` > `assistant` 순으로 정의합니다.

In [12]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6-luna",
    reasoning={"effort": "low"},
    input=[
        {
            "role": "developer",
            "content": "해적처럼 말해 주세요."
        },
        {
            "role": "user",
            "content": "한국의 수도는 부산인가요?"
        }
    ]
)

print(response.output_text)

아니요, 선장님! 한국의 수도는 서울입니다. 부산은 한국의 큰 항구 도시지요.


### 메시지 배열을 쓰는 이유

1. 역할 구분
   - `role: "developer"` 시스템 지시 → 항상 최고 우선순위
   - `role: "user"` 실제 질문
   - `role: "assistant"` 모델의 응답
2. 멀티턴 대화
   - 이전 히스토리를 그대로 쌓아두면 맥락 유지가 쉬움
   - `[developer 지시, user 질문1, assistant 답변1, user 질문2, ...]`

In [13]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6-luna",
    input=[
        {"role": "user", "content": "똑 똑."},
        {"role": "assistant", "content": "누구세요?"},
        {"role": "user", "content": "오렌지."},
    ],
)

print(response.output_text)

오렌지 누구? 🍊


## 5) 멀티턴 대화 구성 실습

### 멀티턴 컨텍스트 관리 기본 개념
- Responses API는 자동으로 대화 히스토리를 보존하지 않습니다.
- 각 턴마다 `role`과 `content` 딕셔너리 및 모델 응답을 히스토리에 추가하여 다음 요청이 히스토리를 공유할 수 있습니다.

In [14]:
# 첫 번째 질문
response = client.responses.create(
    model="gpt-6-luna",
    instructions="You are a helpful assistant. You must answer in Korean.",
    input="대한민국의 수도는 어디인가요?",
)

print(response.output_text)

대한민국의 수도는 서울특별시입니다.


In [15]:
# 두 번째 질문
response = client.responses.create(
    model="gpt-6-luna",
    instructions="You are a helpful assistant. You must answer in Korean.",
    input="방금 답변을 영어로 번역해 주세요",
)

print(response.output_text)

번역할 이전 답변이 대화에 보이지 않습니다. 번역할 내용을 보내 주시면 영어로 옮겨 드릴게요.


In [16]:
context = [
    {
        "role": "developer",
        "content": "You are a helpful assistant. You must answer in Korean.",
    },
    {
        "role": "user",
        "content": "대한민국의 수도는 어디인가요?",
    },
]

res1 = client.responses.create(
    model="gpt-6-luna",
    input=context,
)

first_answer = res1.output_text
print("첫 번째 응답:", first_answer)

첫 번째 응답: 대한민국의 수도는 서울입니다.


In [17]:
context.append({"role": "assistant", "content": first_answer})
context.append({"role": "user", "content": "방금 답변을 영어로 번역해 주세요."})

res2 = client.responses.create(
    model="gpt-6-luna",
    input=context,
)

print("\n두 번째 응답:", res2.output_text)


두 번째 응답: “The capital of the Republic of Korea is Seoul.”


**history 자동 관리를 위한 ask 함수**

In [18]:
def ask(question, message_history=None, *, model="gpt-6-luna", reasoning_effort="medium", verbosity="medium"):
    if message_history is None:
        message_history = [
            {
                "role": "developer",
                "content": "You are a helpful assistant. You must answer in Korean.",
            }
        ]

    message_history.append({"role": "user", "content": question})

    response = client.responses.create(
        model=model,
        input=message_history,
        reasoning={"effort": reasoning_effort},
        text={"verbosity": verbosity},
    )

    message_history.append({"role": "assistant", "content": response.output_text})
    return message_history

In [19]:
# 최초 질문
message_history1 = ask("양자역학에 대해서 쉽게 설명해 주세요")
# 최초 답변
print(message_history1[-1]['content'])

양자역학은 **원자나 전자처럼 아주 작은 세계가 어떻게 움직이는지 설명하는 물리학**입니다. 우리가 매일 보는 공이나 자동차의 움직임과는 조금 다른 규칙을 따릅니다.

핵심은 세 가지예요.

1. **결과를 확률로 예측합니다**  
   공의 위치는 보통 정확히 말할 수 있지만, 전자의 위치는 “여기에 있을 가능성이 높다”처럼 확률로 표현합니다. 측정하기 전에는 여러 가능성을 함께 고려해야 하죠.

2. **측정 전에는 여러 가능성이 겹쳐 있을 수 있습니다**  
   이를 **중첩**이라고 합니다. 동전을 던져 공중에 떠 있는 동안 앞면과 뒷면이 모두 가능한 상태인 것에 비유할 수 있어요. 다만 양자 중첩은 단순히 우리가 결과를 모르는 것보다 더 깊은 물리적 현상입니다.

3. **멀리 떨어진 입자도 서로 연결될 수 있습니다**  
   두 입자가 **얽힘** 상태라면, 한쪽을 측정한 결과와 다른 쪽의 결과가 강하게 연관됩니다. 이 현상은 빛보다 빠르게 정보를 보낼 수 있다는 뜻은 아닙니다.

양자역학은 이상한 이론처럼 들리지만, 실험으로 매우 정확하게 확인됐고 실제 기술에도 쓰입니다. **반도체, 레이저, MRI** 등이 양자역학의 원리를 활용합니다.

한마디로 말하면, 양자역학은 **작은 세계에서는 물체의 상태가 확률적으로 설명되고, 측정 결과에 따라 우리가 관찰하는 모습이 정해진다**는 것을 알려주는 이론입니다.


In [20]:
message_history1

[{'role': 'developer',
  'content': 'You are a helpful assistant. You must answer in Korean.'},
 {'role': 'user', 'content': '양자역학에 대해서 쉽게 설명해 주세요'},
 {'role': 'assistant',
  'content': '양자역학은 **원자나 전자처럼 아주 작은 세계가 어떻게 움직이는지 설명하는 물리학**입니다. 우리가 매일 보는 공이나 자동차의 움직임과는 조금 다른 규칙을 따릅니다.\n\n핵심은 세 가지예요.\n\n1. **결과를 확률로 예측합니다**  \n   공의 위치는 보통 정확히 말할 수 있지만, 전자의 위치는 “여기에 있을 가능성이 높다”처럼 확률로 표현합니다. 측정하기 전에는 여러 가능성을 함께 고려해야 하죠.\n\n2. **측정 전에는 여러 가능성이 겹쳐 있을 수 있습니다**  \n   이를 **중첩**이라고 합니다. 동전을 던져 공중에 떠 있는 동안 앞면과 뒷면이 모두 가능한 상태인 것에 비유할 수 있어요. 다만 양자 중첩은 단순히 우리가 결과를 모르는 것보다 더 깊은 물리적 현상입니다.\n\n3. **멀리 떨어진 입자도 서로 연결될 수 있습니다**  \n   두 입자가 **얽힘** 상태라면, 한쪽을 측정한 결과와 다른 쪽의 결과가 강하게 연관됩니다. 이 현상은 빛보다 빠르게 정보를 보낼 수 있다는 뜻은 아닙니다.\n\n양자역학은 이상한 이론처럼 들리지만, 실험으로 매우 정확하게 확인됐고 실제 기술에도 쓰입니다. **반도체, 레이저, MRI** 등이 양자역학의 원리를 활용합니다.\n\n한마디로 말하면, 양자역학은 **작은 세계에서는 물체의 상태가 확률적으로 설명되고, 측정 결과에 따라 우리가 관찰하는 모습이 정해진다**는 것을 알려주는 이론입니다.'}]

In [21]:
# 두 번째 질문
message_history2 = ask(
    "이전의 내용을 영어로 답변해 주세요", message_history=message_history1
)
# 두 번째 답변
print(message_history2[-1]['content'])

양자역학은 원자나 전자처럼 아주 작은 세계를 설명하는 물리학입니다. 이 세계에서는 입자의 위치나 움직임을 확정적으로 예측하기보다 확률로 설명하는 경우가 많습니다.

측정하기 전에는 여러 가능성이 함께 존재할 수 있는데, 이를 **중첩**이라고 합니다. 또 두 입자가 **얽힘** 상태라면, 서로 멀리 떨어져 있어도 측정 결과가 강하게 연관될 수 있습니다.

양자역학은 낯설게 느껴질 수 있지만 실험으로 확인된 이론이며, 반도체와 레이저, MRI 같은 기술의 바탕이 됩니다.


### previous_response_id를 활용한 멀티턴 대화 예시
- `previous_response_id`를 통해 “대화 상태”를 이어서 활용할 수 있게 합니다.
- 첫 응답의 `response.id`를 다음 요청의 `previous_response_id`로 넘겨 문맥을 이어 줍니다.
- system instruction은 자동으로 이어지지 않습니다.

In [22]:
context = [
    {"role": "developer", "content": "You are a helpful assistant. You must answer in Korean."},
    {"role": "user", "content": "재밌는 농담 해줘."},
]

res1 = client.responses.create(
    model="gpt-6-luna",
    input=context,
)

print(res1.output_text)

수학책이 왜 우울했을까요?

문제가 너무 많아서요. 😄


In [23]:
# 이전 응답의 response_id를 활용하여 대화 이어가기
res2 = client.responses.create(
    model="gpt-6-luna",
    input=[{"role": "user", "content": "그게 왜 재밌는건지 설명해줘."}],
    previous_response_id=res1.id,
)

print(res2.output_text)

‘문제’가 수학책에 나오는 **연습문제**이기도 하고, 사람을 괴롭히는 **골칫거리**라는 뜻도 있어서요. 수학책이 문제로 가득해 우울하다는 말장난이에요.

다만… 설명을 듣고 나니 농담이 더 우울해졌네요. 😅


## 6) Prompting

### Prompting이란?
- Prompting은 모델에 입력을 제공하는 과정입니다.
- 모델 출력 품질은 프롬프트 구성에 크게 좌우됩니다.
- 반복적인 trial&error를 통해 최적의 프롬프트를 찾아가는 것을 권장드립니다.

### (참고) Prompts in the API → 2026년 11월 30일 종료

- 2025년 수업 때는 Dashboard(Playground)에서 프롬프트를 만들고 버전 관리한 뒤, `prompt={"id": "pmpt_...", "version": "1", "variables": {...}}` 형태로 불러오는 **재사용 프롬프트(Prompt 객체)** 를 다뤘습니다.
- 그런데 OpenAI가 이 기능을 deprecated 처리해서 `v1/prompts`는 **2026년 11월 30일에 종료**됩니다.
- 대신 프롬프트는 **애플리케이션 코드에서 직접 관리**하고 `instructions` / `input`으로 넘기는 방식을 권장합니다.
- `{{variable}}` 같은 변수 치환은 아래처럼 Python 문자열 포맷으로 충분합니다.

In [24]:
PROMPT_TEMPLATE = "{country}의 수도 {city}에서 꼭 가봐야 할 장소 3곳을 추천해 주세요."

response = client.responses.create(
    model="gpt-6-luna",
    instructions="You are a friendly travel guide. Reply in Korean.",
    input=PROMPT_TEMPLATE.format(city="서울", country="대한민국"),
)

print(response.output_text)

서울의 역사와 풍경, 전통을 두루 느낄 수 있는 세 곳을 추천해요.

1. **경복궁** — 조선 왕조의 대표 궁궐로, 웅장한 건축물과 아름다운 정원을 둘러보기 좋아요. 한복을 입고 방문하면 특별한 사진도 남길 수 있습니다.
2. **북촌한옥마을** — 전통 한옥이 이어진 고즈넉한 동네예요. 실제 주민들이 사는 곳이니 조용히 둘러보고, 사생활을 존중해 주세요.
3. **남산서울타워** — 서울 시내를 한눈에 내려다볼 수 있는 전망 명소입니다. 특히 해 질 무렵이나 밤에 가면 멋진 야경을 즐길 수 있어요.

경복궁과 북촌은 가까워 함께 둘러보기 좋고, 남산은 저녁 일정으로 넣으면 알찬 하루 코스가 됩니다.


### Prompt Engineering
- Prompt engineering은 프롬프트를 설계해 모델 출력 품질을 향상시키는 전략입니다.
- 모델은 비결정적 → 동일 프롬프트라도 출력이 달라질 수 있습니다.
- 다양한 기법을 적용해 일관된 품질을 얻을 수 있습니다.
- 모델 스냅샷 간에도 결과가 달라질 수 있습니다.

### Markdown & XML을 활용한 Prompt 포맷팅
Markdown 포맷과 XML 태그를 함께 사용하면 프롬프트의 논리적 경계를 더 명확하게 전달할 수 있습니다.
- Markdown → 섹션·계층 구조 표현
- XML → 특정 컨텐츠의 시작/종료 구분, 메타데이터 부여

**Developer 메시지 구성 템플릿**
- Identity: 목적, 말투, 행동 원칙, 목표 등 설정
- Instructions: 어떤 방식으로 답변해야 하는지 규칙과 요구사항 제시
    - 무엇을 해야 하는가?
    - 무엇을 절대로 하면 안 되는가?
    - 어떤 방식으로 함수를 호출해야 하는가?
- Examples: 입력과 출력 예시를 제공
- Context: 추가 정보(사전 지식, 비공개 문서 등)를 제공

In [25]:
developer_message = """
# Identity

You are coding assistant that helps enforce the use of snake case
variables in JavaScript code, and writing code that will run in
Internet Explorer version 6.

# Instructions

* When defining variables, use snake case names (e.g. my_variable)
  instead of camel case names (e.g. myVariable).
* To support old browsers, declare variables using the older
  "var" keyword.
* Do not give responses with Markdown formatting, just return
  the code as requested.

# Examples

<user_query>
How do I declare a string variable for a first name?
</user_query>

<assistant_response>
var first_name = "Anna";
</assistant_response>
"""

response = client.responses.create(
    model="gpt-6-luna",
    instructions=developer_message,
    input="How do I declare a number variable for a user's age?",
)

print(response.output_text)

var user_age = 30;


### Few-shot Learning
- 모델에게 원하는 입력/출력 패턴을 프롬프트로 학습시켜 새로운 task로 유도하는 기법입니다.
- 일반적으로 developer 메시지 내부에 몇 개의 입력/출력 예시를 포함시킵니다.
- Tip:
    - 가능한 다양한 입력 케이스 제공.
    - 각각에 대한 원하는 출력 형식을 함께 포함.

In [26]:
developer_message = """
# Identity

You are a helpful assistant that labels short product reviews as
Positive, Negative, or Neutral.

# Instructions

* Only output a single word in your response with no additional formatting
  or commentary.
* Your response should only be one of the words "Positive", "Negative", or
  "Neutral" depending on the sentiment of the product review you are given.

# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap.
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again.
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
"""

reviews = [
    "배송이 엄청 빨라서 놀랐어요. 포장도 꼼꼼했어요!",
    "가격 대비 그냥 그래요. 나쁘진 않은데 또 살지는 모르겠네요.",
    "이틀 만에 고장 났어요. 환불 요청합니다.",
]

for review in reviews:
    response = client.responses.create(
        model="gpt-6-luna",
        instructions=developer_message,
        input=f"<product_review>\n{review}\n</product_review>",
    )
    print(f"{response.output_text:<8} ← {review}")

Positive ← 배송이 엄청 빨라서 놀랐어요. 포장도 꼼꼼했어요!


Neutral  ← 가격 대비 그냥 그래요. 나쁘진 않은데 또 살지는 모르겠네요.


Negative ← 이틀 만에 고장 났어요. 환불 요청합니다.


**OpenAI Prompting Guideline**

- Prompt engineering 가이드: https://platform.openai.com/docs/guides/prompt-engineering
- (참고) GPT-5 Prompting Guide: https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide

### 참고 자료
- https://platform.openai.com/docs
- https://platform.openai.com/docs/guides/text
- https://platform.openai.com/docs/guides/reasoning